# SpotMicro Kinematics Demo

Body-CoG based inverse kinematics: given a desired body pose `(x, y, z, roll, pitch, yaw)`,
the four legs solve geometrically to accommodate that posture with feet planted on the ground.

The notebook then simulates the transition from an initial pose to a target pose in MuJoCo.

---

**macOS note:** MuJoCo's passive viewer requires `mjpython` instead of the standard Python
interpreter. Run this notebook with:
```
mjpython -m jupyter notebook kinematics_demo.ipynb
```
or execute the simulation cells as a script:
```
mjpython -c "import nbformat; ..."
```
On Linux the standard `jupyter notebook` works fine.


In [ ]:
import sys
import os
import time

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# Notebook lives in mujuco/kinematics/ — parent is mujuco/
MUJUCO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if MUJUCO_DIR not in sys.path:
    sys.path.insert(0, MUJUCO_DIR)

from spotmicro_kinematics import (
    SpotMicroKinematics,
    LEG_NAMES,
    JOINT_LIMITS,
    DEFAULT_HEIGHT,
)
from spotmicro_loader import load_spotmicro_xml

print("Imports OK")

## 1 · Instantiate the kinematics model

In [ ]:
kin = SpotMicroKinematics()
print(kin)

## 2 · Forward kinematics verification at zero joints

In [ ]:
body_pos_zero   = np.array([0.0, 0.0, DEFAULT_HEIGHT])
body_euler_zero = np.array([0.0, 0.0, 0.0])
joints_zero     = np.zeros(12)

feet_fk = kin.forward_kinematics(body_pos_zero, body_euler_zero, joints_zero)

print("FK at zero joints (toe world positions):")
for name, foot in zip(LEG_NAMES, feet_fk):
    print(f"  {name:15s}  x={foot[0]:+.4f}  y={foot[1]:+.4f}  z={foot[2]:+.4f}")

## 3 · IK round-trip test (FK → IK → FK)

In [ ]:
# Use a body height where feet CAN reach z=0 (requires height ≤ L1+L2 ≈ 0.240 m)
STANCE_HEIGHT = 0.200   # natural flex height used for the round-trip test

stance_pos   = np.array([0.0, 0.0, STANCE_HEIGHT])
stance_euler = np.array([0.0, 0.0, 0.0])

stance_feet = kin.compute_stance_feet(stance_pos, stance_euler)
print("Stance feet (z should be ~0.0):")
for name, foot in zip(LEG_NAMES, stance_feet):
    print(f"  {name:15s}  x={foot[0]:+.4f}  y={foot[1]:+.4f}  z={foot[2]:+.4f}")

joints_ik, reachable = kin.inverse_kinematics(stance_pos, stance_euler, stance_feet)
feet_reconstructed   = kin.forward_kinematics(stance_pos, stance_euler, joints_ik)

max_error_mm = np.max(np.abs(stance_feet - feet_reconstructed)) * 1000
print(f"\nFK→IK→FK round-trip max error: {max_error_mm:.4f} mm")
assert max_error_mm < 0.5, f"IK round-trip tolerance exceeded: {max_error_mm:.4f} mm"

print("\nIK joint angles at stance:")
joint_names_flat = [f"{leg[:2].upper()}_{jt}" for leg in LEG_NAMES
                    for jt in ["sh", "hip", "knee"]]
for name, angle in zip(joint_names_flat, joints_ik):
    print(f"  {name:10s}  {np.degrees(angle):+7.2f}°  ({angle:+.4f} rad)")
print(f"\nAll legs reachable: {reachable.all()}")

## 4 · Define start and target poses

Edit these parameters to explore different body poses.

In [ ]:
# ── Start pose ────────────────────────────────────────────────────────────────
# Body height must be ≤ L1+L2 ≈ 0.240 m for the legs to reach z=0 (ground).
START_POS   = np.array([0.0,  0.0,  0.200])   # x, y, z (m)  — natural flex height
START_EULER = np.array([0.0,  0.0,  0.0  ])   # roll, pitch, yaw (rad)

# ── Target pose ───────────────────────────────────────────────────────────────
TARGET_POS   = np.array([0.0,  0.0,  0.170])  # lower (crouched), still reachable
TARGET_EULER = np.array([0.2,  0.1,  0.0  ])  # roll=0.2 rad (~11°), pitch=0.1 rad (~6°)

# ── Trajectory parameters ─────────────────────────────────────────────────────
N_STEPS = 100    # number of interpolation frames
DT      = 0.05   # seconds per frame → 5 s total transition

print(f"Start :  pos={START_POS}  euler={np.degrees(START_EULER)} deg")
print(f"Target:  pos={TARGET_POS}  euler={np.degrees(TARGET_EULER)} deg")
print(f"Duration: {N_STEPS * DT:.1f} s  ({N_STEPS} steps × {DT*1000:.0f} ms)")

## 5 · Generate IK trajectory

In [ ]:
joint_traj, foot_traj, reachable_mask = kin.interpolate_poses(
    START_POS, START_EULER,
    TARGET_POS, TARGET_EULER,
    n_steps=N_STEPS,
    keep_feet_fixed=True,   # feet stay planted — body adjusts over them
)

unreachable = (~reachable_mask.all(axis=1)).sum()
print(f"Trajectory generated: {N_STEPS} steps")
print(f"Unreachable frames: {unreachable} / {N_STEPS}")
print(f"Joint angle range  min={np.degrees(joint_traj.min()):+.1f}°  max={np.degrees(joint_traj.max()):+.1f}°")

## 6 · Plot: joint angles over time

In [ ]:
JOINT_LABELS = ["Shoulder", "Hip", "Knee"]
colors = ["steelblue", "darkorange", "seagreen", "crimson"]
t_axis = np.linspace(0, N_STEPS * DT, N_STEPS)

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True)
fig.suptitle("Joint Angles Along Kinematic Trajectory", fontsize=14)

limit_keys = ["shoulder", "hip", "knee"]

for leg_idx, (leg_name, color) in enumerate(zip(LEG_NAMES, colors)):
    for jt_idx, (jt_label, lk) in enumerate(zip(JOINT_LABELS, limit_keys)):
        ax = axes[jt_idx, leg_idx]
        col = leg_idx * 3 + jt_idx
        ax.plot(t_axis, np.degrees(joint_traj[:, col]), color=color, linewidth=1.5)

        lo, hi = JOINT_LIMITS[lk]
        ax.axhline(np.degrees(lo), color="red",  linestyle="--", linewidth=0.8, alpha=0.6)
        ax.axhline(np.degrees(hi), color="red",  linestyle="--", linewidth=0.8, alpha=0.6)

        if jt_idx == 0:
            ax.set_title(leg_name.replace("_", " ").title(), color=color, fontsize=10)
        if leg_idx == 0:
            ax.set_ylabel(f"{jt_label} (°)", fontsize=9)
        if jt_idx == 2:
            ax.set_xlabel("Time (s)", fontsize=9)
        ax.grid(True, linewidth=0.4)

plt.tight_layout()
plt.show()

## 7 · Plot: foot positions (should stay fixed at z=0)

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
ax.set_title("Foot world positions during trajectory (feet stay planted)")

for leg_idx, (leg_name, color) in enumerate(zip(LEG_NAMES, colors)):
    traj = foot_traj[:, leg_idx, :]
    ax.plot(traj[:, 0], traj[:, 1], traj[:, 2],
            label=leg_name.replace("_", " ").title(), color=color, linewidth=1.5)
    ax.scatter(*traj[0],  marker="o", color=color, s=60, zorder=5)   # start
    ax.scatter(*traj[-1], marker="x", color=color, s=80, zorder=5)   # end

ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.set_zlabel("Z (m)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 8 · Build MuJoCo qpos trajectory

In [ ]:
# Interpolate body poses in parallel with the joint trajectory
q_start = kin._euler_to_quat_wxyz(START_EULER)
q_end   = kin._euler_to_quat_wxyz(TARGET_EULER)

qpos_traj = []
for i in range(N_STEPS):
    t = i / max(N_STEPS - 1, 1)
    body_pos_t   = (1.0 - t) * START_POS + t * TARGET_POS
    body_euler_t = kin._quat_wxyz_to_euler(kin._slerp(q_start, q_end, t))
    qpos_traj.append(kin.build_qpos(body_pos_t, body_euler_t, joint_traj[i]))

qpos_traj = np.array(qpos_traj)   # (N, 19)
print(f"qpos_traj shape: {qpos_traj.shape}")
print(f"qpos[0][:7] (start body): {qpos_traj[0, :7]}")
print(f"qpos[-1][:7] (end body) : {qpos_traj[-1, :7]}")

## 9 · Kinematic mode MuJoCo playback

Directly writes `data.qpos` each step and calls `mj_forward` (geometry update only,
no physics step). This is a pure kinematic replay of the IK trajectory.

> **Tip:** Press **2** in the viewer to show mesh visuals; press **1** to toggle
> collision boxes. Use `v.opt.geomgroup` to control visibility programmatically.

In [ ]:
import mujoco
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

xml_str = load_spotmicro_xml()
model   = mujoco.MjModel.from_xml_string(xml_str)
data    = mujoco.MjData(model)

# ── Render each frame offscreen ───────────────────────────────────────────────
HEIGHT, WIDTH = 480, 640
renderer = mujoco.Renderer(model, height=HEIGHT, width=WIDTH)

frames = []
for qpos_t in qpos_traj:
    data.qpos[:] = qpos_t
    data.qvel[:] = 0.0
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, camera="side_camera")
    frames.append(renderer.render().copy())

renderer.close()
print(f"Rendered {len(frames)} frames")

# ── Display as inline animation ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.axis("off")
img_plot = ax.imshow(frames[0])
plt.tight_layout()

def _update(i):
    img_plot.set_data(frames[i])
    return [img_plot]

anim = animation.FuncAnimation(
    fig, _update,
    frames=len(frames),
    interval=DT * 1000,   # ms per frame
    blit=True,
)
plt.close(fig)   # prevent static duplicate display
HTML(anim.to_jshtml())

## 10 · (Optional) Physics mode: drive to target via joint control

Resets the robot to the start pose, then uses `data.ctrl` (motor targets) to
drive joints toward the IK solution of the target pose. Physics simulates the
body response — the robot will physically balance or fall depending on the pose.

In [ ]:
# Re-use the same model; fresh MjData
data_phys = mujoco.MjData(model)
data_phys.qpos[:] = qpos_traj[0]
data_phys.qvel[:] = 0.0
mujoco.mj_forward(model, data_phys)

target_joints = joint_traj[-1].copy()

# ── Physics simulation — capture frames offscreen ─────────────────────────────
N_PHYS  = 600     # 600 × 0.005 s = 3 s of physics
SKIP    = 5       # render every 5th step to keep frame count manageable

renderer_phys = mujoco.Renderer(model, height=HEIGHT, width=WIDTH)
phys_frames = []

for step in range(N_PHYS):
    data_phys.ctrl[:] = target_joints
    mujoco.mj_step(model, data_phys)
    if step % SKIP == 0:
        renderer_phys.update_scene(data_phys, camera="side_camera")
        phys_frames.append(renderer_phys.render().copy())

renderer_phys.close()
print(f"Rendered {len(phys_frames)} physics frames")

# ── Inline animation ──────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.axis("off")
img2 = ax2.imshow(phys_frames[0])
plt.tight_layout()

def _update2(i):
    img2.set_data(phys_frames[i])
    return [img2]

anim2 = animation.FuncAnimation(
    fig2, _update2,
    frames=len(phys_frames),
    interval=model.opt.timestep * SKIP * 1000,
    blit=True,
)
plt.close(fig2)
HTML(anim2.to_jshtml())